In [1]:
%load_ext autoreload
%autoreload 2
from pathlib import Path
from functools import reduce
from typing import Sequence

import numpy as np
import pandas as pd

In [2]:
RESULTS_PATH = Path("projmix_results_raw_long.xlsx")

In [3]:
df = pd.read_excel(RESULTS_PATH, engine="openpyxl")
df = df.drop(columns="Unnamed: 0")

In [4]:
df.columns

Index(['Seed', 'Model', 'Model Type', 'Metric', 'Sample', 'Noise',
       'Experiment', 'Value', 'Dimension', 'Separability', 'Dependence', 'K',
       'L'],
      dtype='object')

In [5]:
df.head()

,Seed,Model,Model Type,Metric,Sample,Noise,Experiment,Value,Dimension,Separability,Dependence,K,L
0,0,all,all,sample_cross_cov,training,no noise,2d-low-strong,0.28,2,low,strong,0,0
1,0,3-Cylindrical Mixture,cylmix,training_time,training,no noise,2d-low-strong,811.98,2,low,strong,3,0
2,0,3-Cylindrical Mixture,cylmix,em_iters,training,no noise,2d-low-strong,57.00,2,low,strong,3,0
3,0,3-Independent Cylindrical Mixture,indcylmix,training_time,training,no noise,2d-low-strong,346.74,2,low,strong,3,0
4,0,3-Independent Cylindrical Mixture,indcylmix,em_iters,training,no noise,2d-low-strong,8.00,2,low,strong,3,0


In [6]:
def as_list(value):
    return [value] if isinstance(value, str) else list(value)


def metric_spec(metric, sample, prefix, summary, lower_is_better=True):
    return {
        "metric": metric,
        "sample": sample,
        "prefix": prefix,
        "summary": summary,
        "lower_is_better": lower_is_better,
    }


COMPARISON_SPECS = [
    metric_spec("avg_ll", "out of sample", "avg_ll", "diff"),
    metric_spec("bic", "in sample", "bic", "win", lower_is_better=True),
    metric_spec("aic", "in sample", "aic", "win", lower_is_better=True),
    metric_spec("avg_ll", "out of sample", "hll", "win", lower_is_better=False),
    metric_spec("ari", "out of sample", "ari", "diff"),
    metric_spec("training_time", "training", "time", "ratio"),
    metric_spec("em_iters", "training", "em_iters", "ratio"),
]

AVG_LL_COLUMNS = {"avg_ll_mean", "avg_ll_std"}
PERCENT_COLUMNS = {"ari_mean"}


def is_win_rate_column(column):
    return isinstance(column, str) and column.endswith("_win_rate")


def is_time_or_em_iters_column(column):
    return isinstance(column, str) and ("time" in column or "em_iters" in column)


def metric_decimals(column, default=None):
    if column in AVG_LL_COLUMNS:
        return 5
    if column in PERCENT_COLUMNS or is_win_rate_column(column):
        return 2
    if is_time_or_em_iters_column(column):
        return 3
    return default


def round_metric_columns(table, default=None, skip_columns=()):
    rounded = table.copy()
    skip_columns = set(skip_columns)

    for column in rounded.columns:
        if column in skip_columns or not pd.api.types.is_numeric_dtype(rounded[column]):
            continue

        decimals = metric_decimals(column, default=default)
        if decimals is not None:
            rounded[column] = rounded[column].round(decimals)

    return rounded


def format_metric_table(table):
    formatted = table.copy()
    percent_columns = [
        column
        for column in formatted.columns
        if (column in PERCENT_COLUMNS or is_win_rate_column(column))
        and pd.api.types.is_numeric_dtype(formatted[column])
    ]

    if percent_columns:
        formatted[percent_columns] = formatted[percent_columns].mul(100)

    return round_metric_columns(formatted)


def format_metric_values(table, source_column):
    formatted = table.copy()

    if source_column in PERCENT_COLUMNS or is_win_rate_column(source_column):
        formatted = formatted.mul(100)

    decimals = metric_decimals(source_column)
    if decimals is not None:
        formatted = formatted.round(decimals)

    return formatted


def metric_view(df, metric, sample, model1, model2, model_col="Model Type"):
    samples = as_list(sample)

    metric_df = df[
        (df["Metric"] == metric)
        & (df["Sample"].isin(samples))
        & (df[model_col].isin([model1, model2]))
    ]

    return metric_df


def paired_model_values(metric_df, model1, model2, match_cols, model_col="Model Type"):
    paired = metric_df.pivot_table(
        index=match_cols,
        columns=model_col,
        values="Value",
        aggfunc="mean",
    )

    missing = [model for model in [model1, model2] if model not in paired.columns]
    if missing:
        available = paired.columns.to_list()
        raise KeyError(
            f"Missing paired columns for {model_col}: {missing}. "
            f"Available columns: {available}"
        )

    paired = paired.dropna(subset=[model1, model2]).reset_index()
    paired.columns.name = None
    paired["diff"] = paired[model1] - paired[model2]
    paired["ratio"] = paired[model1] / paired[model2]

    return paired


def summarize_diff(
    metric_df,
    model1,
    model2,
    match_cols,
    prefix,
    groupby=None,
    model_col="Model Type",
):
    groupby = ["Dependence"] if groupby is None else groupby

    paired = paired_model_values(
        metric_df, model1, model2, match_cols, model_col=model_col
    )

    return (
        paired
        .groupby(groupby, as_index=False)
        .agg(
            **{
                f"{prefix}_mean": ("diff", "mean"),
                f"{prefix}_std": ("diff", "std"),
            }
        )
    )


def summarize_ratio(
    metric_df,
    model1,
    model2,
    match_cols,
    prefix,
    groupby=None,
    model_col="Model Type",
):
    groupby = ["Dependence"] if groupby is None else groupby

    paired = paired_model_values(
        metric_df, model1, model2, match_cols, model_col=model_col
    )

    return (
        paired
        .groupby(groupby, as_index=False)
        .agg(
            **{
                f"{prefix}_mean": ("ratio", "mean"),
                f"{prefix}_std": ("ratio", "std"),
            }
        )
    )


def summarize_wins(
    metric_df,
    model1,
    model2,
    match_cols,
    prefix,
    groupby=None,
    lower_is_better=True,
    model_col="Model Type",
):
    groupby = ["Dependence"] if groupby is None else groupby

    paired = paired_model_values(
        metric_df, model1, model2, match_cols, model_col=model_col
    )

    if lower_is_better:
        paired["win"] = paired[model1] < paired[model2]
    else:
        paired["win"] = paired[model1] > paired[model2]

    return (
        paired
        .groupby(groupby, as_index=False)
        .agg(
            **{
                f"{prefix}_win_rate": ("win", "mean"),
                f"{prefix}_win_count": ("win", "sum"),
            }
        )
    )


def summarize_metric(df, spec, model1, model2, match_cols, groupby, model_col="Model Type"):
    metric_df = metric_view(
        df,
        spec["metric"],
        spec["sample"],
        model1,
        model2,
        model_col=model_col,
    )
    summary = spec["summary"]

    if summary == "diff":
        return summarize_diff(
            metric_df,
            model1,
            model2,
            match_cols,
            spec["prefix"],
            groupby,
            model_col=model_col,
        )
    if summary == "ratio":
        return summarize_ratio(
            metric_df,
            model1,
            model2,
            match_cols,
            spec["prefix"],
            groupby,
            model_col=model_col,
        )
    if summary == "win":
        return summarize_wins(
            metric_df,
            model1,
            model2,
            match_cols,
            spec["prefix"],
            groupby,
            lower_is_better=spec.get("lower_is_better", True),
            model_col=model_col,
        )

    raise ValueError(f"Unknown summary type: {summary}")


def merge_table_parts(table_parts, groupby):
    return reduce(
        lambda left, right: left.merge(right, on=as_list(groupby), how="outer"),
        table_parts,
    )


def comparison_table(
    df,
    model1,
    model2,
    match_cols,
    groupby,
    specs,
    model_col="Model Type",
    format_output=True,
):
    table_parts = [
        summarize_metric(
            df,
            spec,
            model1,
            model2,
            match_cols,
            groupby,
            model_col=model_col,
        )
        for spec in specs
    ]

    table = merge_table_parts(table_parts, groupby)
    if format_output:
        table = format_metric_table(table)

    return table


def metric_win_rate_pivot(
    df,
    metric,
    sample,
    model1,
    model2,
    match_cols,
    index,
    pivot_column,
    model_col="Model Type",
    lower_is_better=True,
    index_order=None,
    column_order=None,
    scale=100,
    decimals=2,
):
    metric_df = metric_view(
        df, metric, sample, model1, model2, model_col=model_col
    )
    paired = paired_model_values(
        metric_df, model1, model2, match_cols, model_col=model_col
    )

    if lower_is_better:
        paired["win"] = paired[model1] < paired[model2]
    else:
        paired["win"] = paired[model1] > paired[model2]

    table = (
        paired
        .groupby([index, pivot_column], as_index=False)
        .agg(win_rate=("win", "mean"))
        .pivot(index=index, columns=pivot_column, values="win_rate")
    )

    if index_order is not None or column_order is not None:
        table = table.reindex(index=index_order, columns=column_order)
    if scale is not None:
        table = table.mul(scale)
    if decimals is not None:
        table = table.round(decimals)

    return table.reset_index()


def metric_diff_pivot(
    df,
    metric,
    sample,
    model1,
    model2,
    match_cols,
    index,
    pivot_column,
    model_col="Model Type",
    value_name="diff_mean",
    aggfunc="mean",
    index_order=None,
    column_order=None,
):
    metric_df = metric_view(
        df, metric, sample, model1, model2, model_col=model_col
    )
    paired = paired_model_values(
        metric_df, model1, model2, match_cols, model_col=model_col
    )

    table = (
        paired
        .groupby([index, pivot_column], as_index=False)
        .agg(**{value_name: ("diff", aggfunc)})
        .pivot(index=index, columns=pivot_column, values=value_name)
    )

    if index_order is not None or column_order is not None:
        table = table.reindex(index=index_order, columns=column_order)

    table = format_metric_values(table, value_name)

    return table.reset_index()


def ari_recovery_table(source_df, scale=100, decimals=2):
    ari_table_df = source_df[
        (source_df["Metric"] == "ari")
        & (source_df["Sample"] == "out of sample")
    ].copy()

    ari_table_df["Model Label"] = np.select(
        [
            ari_table_df["Model Type"].eq("cylmix"),
            ari_table_df["Model Type"].eq("indcylmix"),
            ari_table_df["Model Type"].eq("mom"),
            ari_table_df["Model Type"].eq("isomom"),
        ],
        [
            "Cylindrical",
            "Independent cylindrical",
            "Full MoM",
            "Isolated MoM",
        ],
        default=None,
    )

    table = (
        ari_table_df
        .dropna(subset=["Model Label"])
        .groupby(["Separability", "Dependence", "Model Label"], as_index=False)
        .agg(ari=("Value", "mean"))
        .pivot_table(
            index=["Separability", "Dependence"],
            columns="Model Label",
            values="ari",
            aggfunc="mean",
        )
        .reindex(
            index=pd.MultiIndex.from_product(
                [["high", "low"], ["ind", "weak", "strong"]],
                names=["Separability", "Dependence"],
            ),
            columns=[
                "Cylindrical",
                "Independent cylindrical",
                "Full MoM",
                "Isolated MoM",
            ],
        )
        .rename(
            index={
                "high": "High",
                "low": "Low",
                "ind": "Conditionally independent",
                "weak": "Weak",
                "strong": "Strong",
            }
        )
    )

    if scale is not None:
        table = table.mul(scale)
    if decimals is not None:
        table = table.round(decimals)

    table = table.reset_index()
    table.columns.name = None

    return table


### Does the cylindrical cross-dependence parameter matter?

In [7]:
model1 = "cylmix"
model2 = "indcylmix"

match_cols_cyl = [
    "Seed",
    "K",
    "Noise",
    "Experiment",
    "Dimension",
    "Separability",
    "Dependence",
    "Sample",
]

groupby_cyl = ["Dependence"]

table_cyl_dependence = comparison_table(
    df,
    model1,
    model2,
    match_cols_cyl,
    groupby_cyl,
    COMPARISON_SPECS,
)


In [8]:
table_cyl_dependence[['Dependence', 'avg_ll_mean', 'avg_ll_std', 'bic_win_rate', 'aic_win_rate', 'ari_mean', 'time_mean', 'em_iters_mean']]

,Dependence,avg_ll_mean,avg_ll_std,bic_win_rate,aic_win_rate,ari_mean,time_mean,em_iters_mean
0,ind,0.00014,0.00264,0.46,39.33,-0.89,1.169,1.124
1,strong,0.27316,0.13629,100.00,100.00,25.81,1.806,3.270
2,weak,0.01744,0.01295,69.29,99.42,1.76,1.201,1.235


In [9]:
table_cyl_bic = metric_win_rate_pivot(
    df,
    "bic",
    "in sample",
    model1,
    model2,
    match_cols_cyl,
    index="Dependence",
    pivot_column="K",
    index_order=["ind", "weak", "strong"],
    column_order=[2, 3, 4],
)

table_cyl_bic


K,Dependence,2,3,4
0,ind,1.38,0.00,0.0
1,weak,100.00,78.88,29.0
2,strong,100.00,100.00,100.0


In [10]:
table_cyl_ari = metric_diff_pivot(
    df,
    "ari",
    ["in sample", "out of sample"],
    model1,
    model2,
    match_cols_cyl,
    index="Dependence",
    pivot_column="K",
    value_name="ari_mean",
    index_order=["ind", "weak", "strong"],
    column_order=[2, 3, 4],
)

table_cyl_ari


K,Dependence,2,3,4
0,ind,-0.10,-1.94,-0.59
1,weak,0.50,1.75,3.02
2,strong,13.35,34.38,29.58


### Does the directional block contain useful information for determining the Euclidean latent class?

In [11]:
model1 = "mom"
model2 = "isomom"

match_cols_mom = [
    "Seed",
    "K",
    "L",
    "Noise",
    "Experiment",
    "Dimension",
    "Separability",
    "Dependence",
    "Sample",
]

groupby_mom = ["Dependence", "L"]

table_mom_dependence = comparison_table(
    df,
    model1,
    model2,
    match_cols_mom,
    groupby_mom,
    COMPARISON_SPECS,
)

table_mom_dependence[['L', 'Dependence', 'avg_ll_mean', 'avg_ll_std', 'bic_win_rate', 'aic_win_rate', 'ari_mean', 'time_mean', 'em_iters_mean']]


,L,Dependence,avg_ll_mean,avg_ll_std,bic_win_rate,aic_win_rate,ari_mean,time_mean,em_iters_mean
0,1,ind,0.00885,0.00902,99.67,99.67,2.72,1.916,0.674
1,2,ind,0.00926,0.00880,99.88,99.88,2.87,1.728,0.479
2,1,strong,0.14096,0.07761,100.00,100.00,-12.41,2.980,1.484
3,2,strong,0.17290,0.07753,100.00,100.00,-12.44,2.955,1.160
4,1,weak,0.01685,0.00962,100.00,100.00,0.56,2.258,0.906
5,2,weak,0.01584,0.00820,100.00,100.00,0.70,1.972,0.602


In [13]:
print(round(table_mom_dependence[table_mom_dependence["Dependence"] == "ind"]["hll_win_count"].sum() / 4800.0 * 100, 2))
print(round(table_mom_dependence[table_mom_dependence["Dependence"] == "weak"]["hll_win_count"].sum() / 4800.0 * 100, 2))
print(round(table_mom_dependence[table_mom_dependence["Dependence"] == "strong"]["hll_win_count"].sum() / 4800.0 * 100, 2))


86.73
99.9
100.0


In [14]:
model1 = "mom"
model2 = "isomom"

match_cols_mom = [
    "Seed",
    "K",
    "L",
    "Noise",
    "Experiment",
    "Dimension",
    "Separability",
    "Dependence",
    "Sample",
]

groupby_mom = ["Separability", "L"]

table_mom_separability = comparison_table(
    df,
    model1,
    model2,
    match_cols_mom,
    groupby_mom,
    COMPARISON_SPECS,
)

table_mom_separability[['L', 'Separability', 'avg_ll_mean', 'avg_ll_std', 'bic_win_rate', 'aic_win_rate', 'ari_mean', 'time_mean', 'em_iters_mean']]


,L,Separability,avg_ll_mean,avg_ll_std,bic_win_rate,aic_win_rate,ari_mean,time_mean,em_iters_mean
0,1,high,0.03614,0.04359,100.00,100.00,-5.79,2.599,1.190
1,2,high,0.04919,0.06332,100.00,100.00,-5.66,2.212,0.792
2,1,low,0.07496,0.09378,99.78,99.78,-0.29,2.171,0.853
3,2,low,0.08281,0.10475,99.92,99.92,-0.25,2.225,0.701


In [15]:
model1 = "mom"
model2 = "isomom"

match_cols_mom = [
    "Seed",
    "K",
    "L",
    "Noise",
    "Experiment",
    "Dimension",
    "Separability",
    "Dependence",
    "Sample",
]

groupby_mom = ["Dependence", "Separability", "L"]

table_mom_separability2 = comparison_table(
    df,
    model1,
    model2,
    match_cols_mom,
    groupby_mom,
    COMPARISON_SPECS,
)

table_mom_separability2[table_mom_separability2.Dependence.eq("ind")][
    ['L', 'Separability', 'avg_ll_mean']
]

,L,Separability,avg_ll_mean
0,1,high,0.01641
1,2,high,0.01680
2,1,low,0.00129
3,2,low,0.00172


### How does the cylindrical mixture model compare against the hierarchical mixture model?

In [16]:
model1 = "3-Cylindrical Mixture"
model2 = "(3,2)-Two-layer MoM"

match_cols_vs = [
    "Seed",
    "Noise",
    "Experiment",
    "Dimension",
    "Separability",
    "Dependence",
    "Sample",
]

groupby_vs = ["Dependence", "Separability"]

table_vs = comparison_table(
    df,
    model1,
    model2,
    match_cols_vs,
    groupby_vs,
    COMPARISON_SPECS,
    model_col="Model",
)

table_vs[['Dependence','Separability','avg_ll_mean','avg_ll_std','bic_win_rate','aic_win_rate','hll_win_rate','ari_mean', 'time_mean','em_iters_mean']]


,Dependence,Separability,avg_ll_mean,avg_ll_std,bic_win_rate,aic_win_rate,hll_win_rate,ari_mean,time_mean,em_iters_mean
0,ind,high,-0.00256,0.00394,12.00,34.75,29.50,-8.89,0.398,0.959
1,ind,low,-0.00049,0.00248,0.75,34.50,42.25,0.56,0.436,1.133
2,strong,high,0.22140,0.09123,100.00,100.00,100.00,63.18,0.605,2.019
3,strong,low,0.24669,0.14179,100.00,100.00,100.00,1.94,0.382,1.023
4,weak,high,0.01630,0.01165,100.00,100.00,98.25,0.14,0.426,1.199
5,weak,low,0.00968,0.00836,98.25,99.75,92.50,0.20,0.358,0.887


### Clustering recovery

In [17]:
ari_table = ari_recovery_table(df)

ari_table

,Separability,Dependence,Cylindrical,Independent cylindrical,Full MoM,Isolated MoM
0,High,Conditionally independent,30.03,31.57,34.72,29.76
1,High,Weak,32.42,29.17,31.71,29.74
2,High,Strong,54.38,5.62,5.68,29.79
3,Low,Conditionally independent,2.43,2.67,1.49,0.87
4,Low,Weak,0.53,0.26,0.16,0.86
5,Low,Strong,2.92,0.06,0.09,0.83


### With-noise-only tables

In [18]:
from IPython.display import display

with_noise_df = df[df["Noise"].eq("with noise")].copy()

table_cyl_dependence_with_noise = comparison_table(
    with_noise_df,
    "cylmix",
    "indcylmix",
    match_cols_cyl,
    ["Dependence"],
    COMPARISON_SPECS,
)

table_mom_dependence_with_noise = comparison_table(
    with_noise_df,
    "mom",
    "isomom",
    match_cols_mom,
    ["Dependence", "L"],
    COMPARISON_SPECS,
)

table_mom_separability_with_noise = comparison_table(
    with_noise_df,
    "mom",
    "isomom",
    match_cols_mom,
    ["Separability", "L"],
    COMPARISON_SPECS,
)

table_vs_with_noise = comparison_table(
    with_noise_df,
    "3-Cylindrical Mixture",
    "(3,2)-Two-layer MoM",
    match_cols_vs,
    ["Dependence", "Separability"],
    COMPARISON_SPECS,
    model_col="Model",
)

ari_table_with_noise = ari_recovery_table(with_noise_df)

with_noise_tables = {
    "table_cyl_dependence": table_cyl_dependence_with_noise[["Dependence", "avg_ll_mean", "avg_ll_std", "bic_win_rate", "aic_win_rate", "ari_mean", "time_mean", "em_iters_mean"]],
    "table_mom_dependence": table_mom_dependence_with_noise[["L", "Dependence", "avg_ll_mean", "avg_ll_std", "bic_win_rate", "aic_win_rate", "time_mean", "em_iters_mean"]],
    "table_mom_separability": table_mom_separability_with_noise[["L", "Separability", "avg_ll_mean", "avg_ll_std", "bic_win_rate", "aic_win_rate", "time_mean", "em_iters_mean"]],
    "table_vs": table_vs_with_noise[["Dependence", "Separability", "avg_ll_mean", "avg_ll_std", "bic_win_rate", "aic_win_rate", "hll_win_rate", "time_mean", "em_iters_mean"]],
    "ari_table": ari_table_with_noise,
}

for name, table in with_noise_tables.items():
    print(f"\n{name}")
    display(table)



table_cyl_dependence


,Dependence,avg_ll_mean,avg_ll_std,bic_win_rate,aic_win_rate,ari_mean,time_mean,em_iters_mean
0,ind,0.00006,0.00256,0.25,37.33,-0.84,1.157,1.122
1,strong,0.25542,0.12919,100.00,100.00,25.14,1.788,3.234
2,weak,0.01675,0.01244,67.92,99.25,1.69,1.189,1.232



table_mom_dependence


,L,Dependence,avg_ll_mean,avg_ll_std,bic_win_rate,aic_win_rate,time_mean,em_iters_mean
0,1,ind,0.00879,0.00904,99.67,99.67,1.898,0.663
1,2,ind,0.00917,0.00879,99.92,99.92,1.717,0.477
2,1,strong,0.13703,0.07524,100.00,100.00,2.951,1.473
3,2,strong,0.16773,0.07464,100.00,100.00,2.950,1.161
4,1,weak,0.01652,0.00942,100.00,100.00,2.242,0.901
5,2,weak,0.01544,0.00791,100.00,100.00,1.958,0.600



table_mom_separability


,L,Separability,avg_ll_mean,avg_ll_std,bic_win_rate,aic_win_rate,time_mean,em_iters_mean
0,1,high,0.03527,0.04206,100.00,100.00,2.567,1.180
1,2,high,0.04794,0.06114,100.00,100.00,2.225,0.800
2,1,low,0.07295,0.09114,99.78,99.78,2.161,0.845
3,2,low,0.08029,0.10148,99.94,99.94,2.192,0.692



table_vs


,Dependence,Separability,avg_ll_mean,avg_ll_std,bic_win_rate,aic_win_rate,hll_win_rate,time_mean,em_iters_mean
0,ind,high,-0.00249,0.00381,11.0,34.5,30.5,0.383,0.963
1,ind,low,-0.00053,0.00252,0.0,32.0,42.0,0.427,1.139
2,strong,high,0.19951,0.08478,100.0,100.0,100.0,0.597,2.039
3,strong,low,0.23382,0.13456,100.0,100.0,100.0,0.367,1.017
4,weak,high,0.01558,0.01110,100.0,100.0,98.5,0.417,1.215
5,weak,low,0.00952,0.00849,98.0,99.5,90.0,0.353,0.912



ari_table


,Separability,Dependence,Cylindrical,Independent cylindrical,Full MoM,Isolated MoM
0,High,Conditionally independent,29.69,31.13,34.26,29.34
1,High,Weak,31.98,28.87,31.35,29.31
2,High,Strong,53.47,5.58,5.68,29.35
3,Low,Conditionally independent,2.42,2.65,1.44,0.85
4,Low,Weak,0.53,0.27,0.16,0.85
5,Low,Strong,2.45,0.06,0.09,0.82


### Original-minus-noisy table differences

In [19]:
from IPython.display import display

TABLE_ID_COLUMNS = [
    "Separability",
    "Dependence",
    "L",
    "K",
    "Sample",
    "Noise",
    "Experiment",
    "Dimension",
]


def difference_table(original_table, noisy_table, scale_numeric=False):
    label_cols = [
        col
        for col in original_table.columns
        if col in noisy_table.columns and col in TABLE_ID_COLUMNS
    ]
    numeric_cols = [
        col
        for col in original_table.columns
        if col in noisy_table.columns
        and col not in label_cols
        and pd.api.types.is_numeric_dtype(original_table[col])
        and pd.api.types.is_numeric_dtype(noisy_table[col])
    ]

    if label_cols:
        original_indexed = original_table.set_index(label_cols)
        noisy_indexed = noisy_table.set_index(label_cols)
    else:
        original_indexed = original_table.copy()
        noisy_indexed = noisy_table.copy()

    original_values = original_indexed[numeric_cols]
    noisy_values = noisy_indexed[numeric_cols].reindex(original_values.index)
    diff = original_values - noisy_values
    if scale_numeric:
        diff = diff.mul(100)

    if label_cols:
        diff = diff.reset_index()
    diff.columns.name = None
    if scale_numeric:
        diff = round_metric_columns(diff, default=2, skip_columns=label_cols)
    else:
        diff = format_metric_table(diff)

    return diff


original_tables = {
    "table_cyl_dependence": comparison_table(
        df,
        "cylmix",
        "indcylmix",
        match_cols_cyl,
        ["Dependence"],
        COMPARISON_SPECS,
        format_output=False,
    )[["Dependence", "avg_ll_mean", "avg_ll_std", "bic_win_rate", "aic_win_rate", "ari_mean", "time_mean", "em_iters_mean"]],
    "table_mom_dependence": comparison_table(
        df,
        "mom",
        "isomom",
        match_cols_mom,
        ["Dependence", "L"],
        COMPARISON_SPECS,
        format_output=False,
    )[["L", "Dependence", "avg_ll_mean", "avg_ll_std", "bic_win_rate", "aic_win_rate", "time_mean", "em_iters_mean"]],
    "table_mom_separability": comparison_table(
        df,
        "mom",
        "isomom",
        match_cols_mom,
        ["Separability", "L"],
        COMPARISON_SPECS,
        format_output=False,
    )[["L", "Separability", "avg_ll_mean", "avg_ll_std", "bic_win_rate", "aic_win_rate", "time_mean", "em_iters_mean"]],
    "table_vs": comparison_table(
        df,
        "3-Cylindrical Mixture",
        "(3,2)-Two-layer MoM",
        match_cols_vs,
        ["Dependence", "Separability"],
        COMPARISON_SPECS,
        model_col="Model",
        format_output=False,
    )[["Dependence", "Separability", "avg_ll_mean", "avg_ll_std", "bic_win_rate", "aic_win_rate", "hll_win_rate", "time_mean", "em_iters_mean"]],
    "ari_table": ari_recovery_table(df, scale=None, decimals=None),
}

with_noise_tables_raw = {
    "table_cyl_dependence": comparison_table(
        with_noise_df,
        "cylmix",
        "indcylmix",
        match_cols_cyl,
        ["Dependence"],
        COMPARISON_SPECS,
        format_output=False,
    )[["Dependence", "avg_ll_mean", "avg_ll_std", "bic_win_rate", "aic_win_rate", "ari_mean", "time_mean", "em_iters_mean"]],
    "table_mom_dependence": comparison_table(
        with_noise_df,
        "mom",
        "isomom",
        match_cols_mom,
        ["Dependence", "L"],
        COMPARISON_SPECS,
        format_output=False,
    )[["L", "Dependence", "avg_ll_mean", "avg_ll_std", "bic_win_rate", "aic_win_rate", "time_mean", "em_iters_mean"]],
    "table_mom_separability": comparison_table(
        with_noise_df,
        "mom",
        "isomom",
        match_cols_mom,
        ["Separability", "L"],
        COMPARISON_SPECS,
        format_output=False,
    )[["L", "Separability", "avg_ll_mean", "avg_ll_std", "bic_win_rate", "aic_win_rate", "time_mean", "em_iters_mean"]],
    "table_vs": comparison_table(
        with_noise_df,
        "3-Cylindrical Mixture",
        "(3,2)-Two-layer MoM",
        match_cols_vs,
        ["Dependence", "Separability"],
        COMPARISON_SPECS,
        model_col="Model",
        format_output=False,
    )[["Dependence", "Separability", "avg_ll_mean", "avg_ll_std", "bic_win_rate", "aic_win_rate", "hll_win_rate", "time_mean", "em_iters_mean"]],
    "ari_table": ari_recovery_table(with_noise_df, scale=None, decimals=None),
}

difference_tables = {
    name: difference_table(
        original_tables[name],
        with_noise_tables_raw[name],
        scale_numeric=name == "ari_table",
    )
    for name in original_tables
}

for name, table in difference_tables.items():
    print(f"\n{name} (original - noisy)")
    display(table)



table_cyl_dependence (original - noisy)


,Dependence,avg_ll_mean,avg_ll_std,bic_win_rate,aic_win_rate,ari_mean,time_mean,em_iters_mean
0,ind,0.00008,0.00008,0.21,2.00,-0.05,0.013,0.002
1,strong,0.01775,0.00711,0.00,0.00,0.67,0.019,0.036
2,weak,0.00070,0.00051,1.37,0.17,0.08,0.011,0.004



table_mom_dependence (original - noisy)


,L,Dependence,avg_ll_mean,avg_ll_std,bic_win_rate,aic_win_rate,time_mean,em_iters_mean
0,1,ind,0.00006,-0.00003,0.00,0.00,0.018,0.012
1,2,ind,0.00009,0.00001,-0.04,-0.04,0.011,0.002
2,1,strong,0.00393,0.00237,0.00,0.00,0.029,0.011
3,2,strong,0.00517,0.00289,0.00,0.00,0.005,-0.001
4,1,weak,0.00033,0.00020,0.00,0.00,0.016,0.005
5,2,weak,0.00040,0.00029,0.00,0.00,0.013,0.001



table_mom_separability (original - noisy)


,L,Separability,avg_ll_mean,avg_ll_std,bic_win_rate,aic_win_rate,time_mean,em_iters_mean
0,1,high,0.00087,0.00153,0.00,0.00,0.032,0.010
1,2,high,0.00125,0.00218,0.00,0.00,-0.013,-0.008
2,1,low,0.00201,0.00264,0.00,0.00,0.010,0.008
3,2,low,0.00252,0.00327,-0.03,-0.03,0.032,0.009



table_vs (original - noisy)


,Dependence,Separability,avg_ll_mean,avg_ll_std,bic_win_rate,aic_win_rate,hll_win_rate,time_mean,em_iters_mean
0,ind,high,-0.00006,0.00013,1.00,0.25,-1.00,0.015,-0.004
1,ind,low,0.00004,-0.00004,0.75,2.50,0.25,0.009,-0.006
2,strong,high,0.02190,0.00645,0.00,0.00,0.00,0.009,-0.020
3,strong,low,0.01288,0.00723,0.00,0.00,0.00,0.014,0.006
4,weak,high,0.00072,0.00055,0.00,0.00,-0.25,0.009,-0.016
5,weak,low,0.00016,-0.00013,0.25,0.25,2.50,0.005,-0.025



ari_table (original - noisy)


,Separability,Dependence,Cylindrical,Independent cylindrical,Full MoM,Isolated MoM
0,High,Conditionally independent,0.35,0.44,0.46,0.42
1,High,Weak,0.44,0.29,0.37,0.43
2,High,Strong,0.91,0.04,-0.00,0.44
3,Low,Conditionally independent,0.02,0.02,0.05,0.01
4,Low,Weak,-0.00,-0.01,-0.00,0.01
5,Low,Strong,0.47,-0.00,0.00,0.02
